# Module 03: Adaptive Pricing via Contextual Bandits  
## Part 1 — Motivation, Assumptions, and Environment

### Objective
To demonstrate **why static pricing fails under non-stationary demand** and motivate the need for **online, safety-aware bandit agents**.

This module builds on:
- **Module 02**: Hierarchical Bayesian Demand Modeling
- Introduces **online adaptation** under uncertainty

### Key Question
> How should a pricing system adapt when demand shifts over time and only sparse, noisy feedback is available?

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")
np.random.seed(42)


## 1. Problem Context & Assumptions

We consider a **single-period pricing problem**:

- At each time step, the system chooses **one price**
- The market returns **binary feedback**: booking or no booking
- Demand depends on price and latent market conditions
- Feedback is **noisy and sparse**

### Explicit Assumptions
- No inventory constraints (handled in later modules)
- No cancellations
- No delayed or long-term rewards
- Independent pricing decisions across time

These assumptions isolate **adaptation behavior** and reduce confounding factors.


## 2. Why Static Demand Models Are Not Enough

In Module 02, we built strong **static demand models** using:
- Hierarchical Bayesian estimation
- Seasonality-aware features
- Neighborhood and listing-level aggregation

However, **even a perfect static model becomes suboptimal when the environment changes**.

### Real-world sources of non-stationarity:
- Seasonal transitions
- Demand shocks (events, holidays, competition)
- User behavior drift
- Macro effects

We demonstrate this failure using a controlled simulation.


## 3. Simulated Market Environment

To study adaptation, we require **interaction**:
- The agent must choose prices
- The market responds stochastically
- The true demand curve is **unknown to the agent**

We therefore simulate a market that represents real user behavior.


In [ ]:
def true_demand_probability(
    price,
    base_price=150,
    elasticity=-0.04,
    shock=0.0
):
    """
    Ground-truth demand probability.
    Unknown to the pricing agent.
    """
    logit = -elasticity * (price - base_price) + shock
    return 1 / (1 + np.exp(-logit))


def simulate_booking(price, **kwargs):
    """
    Simulate binary booking outcome.
    """
    p = true_demand_probability(price, **kwargs)
    return np.random.binomial(1, p)


## 4. Static Optimal Pricing (Oracle Baseline)

Assume an oracle that knows the true demand curve.
The static optimal price is:


$$ p^* = \arg\max_p \; p \cdot q(p) $$


We compute this once and apply it blindly over time.


In [ ]:
prices = np.linspace(50, 300, 300)
true_probs = np.array([true_demand_probability(p) for p in prices])
oracle_revenue = prices * true_probs

static_opt_price = prices[np.argmax(oracle_revenue)]
static_opt_price


## 5. Demand Shock Scenario

We now introduce **non-stationarity**:

- Time horizon: 200 steps
- At time `T = 100`, demand suddenly drops
- The static pricing policy does **not adapt**

This simulates events such as:
- season change
- demand collapse
- market saturation


In [ ]:
T = 200
shock_time = 100

revenues = []

for t in range(T):
    shock = -1.0 if t >= shock_time else 0.0
    booked = simulate_booking(static_opt_price, shock=shock)
    revenues.append(static_opt_price * booked)

cumulative_revenue = np.cumsum(revenues)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(cumulative_revenue, label="Static Pricing")
plt.axvline(shock_time, color="red", linestyle="--", label="Demand Shock")
plt.title("Failure of Static Pricing Under Demand Shock")
plt.xlabel("Time")
plt.ylabel("Cumulative Revenue")
plt.legend()
plt.show()


## 6. Interpretation

Observations:

- Static pricing performs well **only before the shock**
- After the shock, revenue growth slows significantly
- The system has **no mechanism to recover**

### Key Insight
> Static demand models, regardless of accuracy, are insufficient in non-stationary environments.

This motivates **online adaptation**, where pricing decisions:
- learn from feedback
- adapt to change
- balance exploration and exploitation

In **Part 2**, we formalize this as a **contextual bandit problem** and introduce belief modeling.


# Part 2 — Bandit Formulation & Belief Architecture

In Part 1, we showed that static pricing fails under non-stationary demand.

In this section, we:
- formalize pricing as a contextual bandit
- define belief objects used by the agent
- implement prior, online, and fused beliefs in code

⚠️ No policy optimization yet — only belief mechanics.


## 1. Pricing as a Contextual Bandit

At each time step $ t $:

- Observe context $ x_t $
- Choose price $ p_t $
- Observe booking $ y_t \sim \text{Bernoulli}(q(p_t, x_t)) $

Objective:
$$
\max_{p_t} \; \mathbb{E}[p_t \cdot y_t]
$$

Only feedback for the **chosen price** is observed.


In [ ]:
from dataclasses import dataclass

@dataclass
class DemandBelief:
    mean: float        # Expected booking probability
    sigma: float       # Uncertainty (std dev)
    source: str        # 'prior', 'online', or 'fused'


## 2. Prior Belief (Offline Knowledge)

The prior belief comes from the demand model trained in Module 02.

Properties:
- Stable
- Conservative
- High uncertainty at fine granularity

We treat the prior as a black box that returns:
- mean demand
- uncertainty


In [ ]:
def prior_belief(price):
    """
    Mock prior belief.
    Represents output of Hierarchical Bayesian Demand Model.
    """
    mean = 1 / (1 + np.exp(0.03 * (price - 150)))
    sigma = 0.15
    return DemandBelief(mean=mean, sigma=sigma, source="prior")


## 3. Online Belief (Streaming Feedback)

The online belief is updated only from:
- prices actually chosen
- observed booking outcomes

Key properties:
- Starts uninformative
- Adapts quickly
- Uses discounted forgetting


In [ ]:
class SimpleOnlineBelief:
    def __init__(self):
        self.mean = 0.5
        self.sigma = 0.3
        self.n = 0

    def update(self, booked):
        self.n += 1
        lr = 1 / self.n
        self.mean = (1 - lr) * self.mean + lr * booked
        self.sigma = max(0.05, self.sigma * 0.95)

    def predict(self):
        return DemandBelief(
            mean=self.mean,
            sigma=self.sigma,
            source="online"
        )


## 4. Belief Fusion

The agent never trusts the online belief blindly.

Instead, it fuses beliefs using confidence:
- High uncertainty → rely on prior
- Low uncertainty → trust online signal

This prevents overreaction and unsafe pricing.


In [ ]:
def fuse_beliefs(prior: DemandBelief, online: DemandBelief):
    w_p = 1 / (prior.sigma**2 + 1e-6)
    w_o = 1 / (online.sigma**2 + 1e-6)

    mean = (prior.mean * w_p + online.mean * w_o) / (w_p + w_o)
    sigma = np.sqrt(1 / (w_p + w_o))

    return DemandBelief(mean=mean, sigma=sigma, source="fused")


In [ ]:
price = 150

prior = prior_belief(price)
online_model = SimpleOnlineBelief()

print("Initial prior:", prior)

# Simulate a few bookings
for outcome in [1, 0, 1, 1]:
    online_model.update(outcome)

online = online_model.predict()
fused = fuse_beliefs(prior, online)

prior, online, fused


## 5. Interpretation

- The **prior** provides stability
- The **online belief** adapts to evidence
- The **fused belief** balances both

This belief object is what bandit algorithms will use
to make pricing decisions.

In the next part, we introduce concrete bandit
policies that act on these beliefs.


## Next: Decision Rules

With beliefs defined and implemented, we now need
rules to choose prices.

Part 3 introduces:
- Thompson Sampling
- Bayesian UCB
- LinUCB

All policies will consume the **same belief objects**.


# Part 3 — Bandit Algorithms & Decision Rules

In Part 2, we defined **beliefs**:
- Prior belief (offline, conservative)
- Online belief (adaptive, uncertain)
- Fused belief (variance-aware)

In this section, we define **how prices are chosen** from these beliefs.

Key idea:
> Different bandit algorithms differ only in **how they use uncertainty**, not in what they believe.


## 1. Why Multiple Bandit Algorithms?

There is no single "best" exploration strategy.

Different business environments require different trade-offs:

- Aggressive growth → more exploration
- Enterprise pricing → bounded risk
- Debugging / baselines → deterministic behavior

We therefore implement and compare:
1. Thompson Sampling
2. Bayesian UCB
3. LinUCB


In [ ]:
def price_grid(min_p=50, max_p=300, n=40):
    return np.linspace(min_p, max_p, n)


## 2. Shared Decision Interface

All bandit algorithms follow the same interface:

1. Enumerate candidate prices
2. For each price, estimate:
   - mean booking probability
   - uncertainty
3. Apply an exploration rule
4. Select the price with highest score

Only step (3) differs across algorithms.


## 3. Thompson Sampling

**Idea**:
- Sample demand from the belief distribution
- Act optimally under the sampled world

This makes exploration:
- stochastic
- proportional to uncertainty
- naturally decaying as confidence increases

Thompson Sampling is typically revenue-maximizing,
but can be volatile.


In [ ]:
def thompson_decision(prices, means, sigmas):
    """
    Thompson Sampling over candidate prices.
    """
    samples = means + sigmas * np.random.randn(len(means))
    samples = np.clip(samples, 0.01, 0.99)

    revenues = prices * samples
    return prices[np.argmax(revenues)]


## 4. Bayesian Upper Confidence Bound (UCB)

**Idea**:
- Optimistically assume demand is higher than estimated
- Exploration is explicit and bounded

Decision rule:
$$
\text{score}(p) = p \cdot (\mu(p) + \beta \sigma(p))
$$

Properties:
- More conservative than Thompson
- Easier to reason about
- Preferred in risk-sensitive settings


In [ ]:
def bayesian_ucb_decision(prices, means, sigmas, beta=1.5):
    scores = prices * (means + beta * sigmas)
    return prices[np.argmax(scores)]


## 5. LinUCB (Deterministic Baseline)

**Idea**:
- Linear approximation of demand
- Deterministic optimism via confidence bounds

Why include it?
- Interpretable
- Fast
- Strong baseline for comparison

LinUCB helps verify that gains are not due to randomness alone.


In [ ]:
def linucb_decision(prices, means, sigmas, alpha=1.0):
    scores = prices * (means + alpha * sigmas)
    return prices[np.argmax(scores)]


## 6. Comparing Decision Rules

| Algorithm | Uses Uncertainty | Stochastic | Risk Level |
|---------|------------------|------------|------------|
| Thompson | Implicitly | Yes | Medium–High |
| Bayes-UCB | Explicitly | No | Medium |
| LinUCB | Explicitly | No | Low |

All three use the **same beliefs**.
Only the decision rule differs.


In [ ]:
prices = price_grid()

# Mock belief (same for all algorithms)
means = 1 / (1 + np.exp(0.03 * (prices - 150)))
sigmas = np.linspace(0.2, 0.05, len(prices))  # higher uncertainty at extremes

print("Thompson price:", thompson_decision(prices, means, sigmas))
print("Bayesian UCB price:", bayesian_ucb_decision(prices, means, sigmas))
print("LinUCB price:", linucb_decision(prices, means, sigmas))


## 7. Interpretation

Even with identical beliefs:
- Thompson Sampling may choose different prices each run
- UCB chooses a conservative optimistic price
- LinUCB behaves deterministically

This confirms that **adaptation behavior is a policy choice**,
not a modeling artifact.


## Next: Online Simulation & Adaptation

So far:
- Beliefs are defined
- Decision rules are implemented

In Part 4, we:
- connect bandits to the market simulator
- run full interaction loops
- measure adaptation, regret, and safety metrics


# Part 4 — Online Simulation, Shock Adaptation & Model Comparison

In Parts 1–3, we:
- showed why static pricing fails
- formalized pricing as a contextual bandit
- defined belief-based decision rules

In this section, we:
- plug in the **full production bandit system (`bandit.py`)**
- simulate online interaction
- test **adaptation under demand shocks**
- compare **Thompson, Bayesian UCB, and LinUCB**
- analyze **safety behavior**, not just revenue


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from pricing_engine.Bandit import (
    ThompsonPricingBandit,
    BayesianUCBBandit,
    LinUCBBandit
)

from pricing_engine.demand_model import HierarchicalBayesianLogisticDemand


In [ ]:
from pricing_engine.Bandit import ThompsonPricingBandit

from pricing_engine.demand_model import (
    HierarchicalBayesianLogisticDemand,
    SeasonalElasticityDemand,
    NeighborhoodResidualCorrector,
    MonotoneDemandWrapper,
    MonotoneGAMDemand
)

from pricing_engine.data_loader import load_and_clean_seattle_data


## 1. Simulation Philosophy

The bandit **never sees the true demand function**.

Instead:
- the agent chooses a price
- the market returns a binary booking
- the agent updates its belief

We simulate the market to:
- control shocks
- evaluate adaptation speed
- stress-test safety mechanisms

This simulation stands in for **real users**.


In [ ]:
def market_response(price, t, shock_time=100):
    """
    Ground-truth environment (unknown to the agent).
    Demand shifts upward after shock_time.
    """
    if t < shock_time:
        optimal = 100
    else:
        optimal = 180  # demand shock

    sensitivity = 0.05
    logit = -sensitivity * (price - optimal)
    prob = 1 / (1 + np.exp(-logit))
    return np.random.binomial(1, prob)


In [ ]:
context = {
    "listing_id": "L1",
    "day_of_year": 180,
    "dow": 5,
    "is_weekend": 1
}


## 2. Training the Static Prior (Module 02)

We train a hierarchical Bayesian demand model
on historical data.

This model provides:
- initial belief
- structured uncertainty
- cold-start safety


In [ ]:
from pathlib import Path

try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()

PROJECT_ROOT = SCRIPT_DIR.parent

CALENDAR_PATH = PROJECT_ROOT / "data" / "calendar.csv"
LISTINGS_PATH = PROJECT_ROOT / "data" / "listings.csv"

df = load_and_clean_seattle_data(CALENDAR_PATH, LISTINGS_PATH)


In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["day_of_year"] = df["date"].dt.dayofyear
df["dow"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["dow"] >= 5).astype(int)

feature_cols = [
    "day_of_year",
    "dow",
    "is_weekend"
]


In [ ]:
# NOTE: Replace df with your cleaned Seattle data
feature_cols = ["day_of_year", "dow", "is_weekend"]

prior_model = HierarchicalBayesianLogisticDemand(beta_price=1.0)
prior_model.fit(df, feature_cols)

def prior_predict_fn(X):
    """
    Vectorized wrapper required by BasePricingBandit.
    """
    means, stds = [], []
    for x in X:
        ctx = dict(zip(feature_cols, x[:-1]))
        prob = prior_model.predict(ctx, np.expm1(x[-1])).prob
        std = prior_model.predict(ctx, np.expm1(x[-1])).std_dev
        means.append(prob)
        stds.append(std)
    return np.array(means), np.array(stds)


In [ ]:
agents = {
    "Thompson": ThompsonPricingBandit(
        feature_names=feature_cols,
        scaler=prior_model.scaler,
        predict_fn=prior_predict_fn,
        forgetting_factor=0.90
    ),
    "BayesianUCB": BayesianUCBBandit(
        feature_names=feature_cols,
        scaler=prior_model.scaler,
        predict_fn=prior_predict_fn,
        forgetting_factor=0.90
    ),
    "LinUCB": LinUCBBandit(
        feature_names=feature_cols,
        scaler=prior_model.scaler,
        predict_fn=prior_predict_fn,
        forgetting_factor=0.90
    )
}


In [ ]:
def run_simulation(agent, T=200, shock_time=100):
    history = []

    for t in range(T):
        decision = agent.choose_price(context, min_p=50, max_p=250)
        booked = market_response(decision.selected_price, t, shock_time)

        agent.update(context, decision.selected_price, booked)

        history.append({
            "t": t,
            "price": decision.selected_price,
            "booked": booked,
            "panic": decision.panic_mode,
            "sigma": decision.uncertainty_sigma,
        })

    return pd.DataFrame(history)


In [ ]:
results = {}

for name, agent in agents.items():
    print(f"Running simulation for {name}...")
    results[name] = run_simulation(agent)


In [ ]:
plt.figure(figsize=(12, 5))

for name, df in results.items():
    plt.plot(df["t"], df["price"], label=name)

plt.axvline(100, color="red", linestyle="--", label="Demand Shock")
plt.axhline(100, color="blue", linestyle=":", label="Optimal Pre-Shock")
plt.axhline(180, color="green", linestyle=":", label="Optimal Post-Shock")

plt.title("Price Adaptation Under Demand Shock")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.show()


## 3. Safety & Stability Metrics

Beyond revenue, we analyze:

- Price volatility
- Panic frequency
- Recovery time after shock
- Uncertainty collapse


In [ ]:
metrics = []

for name, df in results.items():
    metrics.append({
        "agent": name,
        "avg_price": df["price"].mean(),
        "price_volatility": df["price"].std(),
        "panic_rate": df["panic"].mean(),
        "avg_uncertainty": df["sigma"].mean()
    })

res = pd.DataFrame(metrics)


In [ ]:
res

In [ ]:
for name, df in results.items():
    shock_recovery = df[df["t"] > 100]["price"].mean() - df[df["t"] <= 100]["price"].mean()
    print(name, "price shift after shock:", shock_recovery)


# Part 5 — Safety Layer, Guardrails & Meta-Policy Evaluation

In Part 4, we observed:

- Bandits successfully adapt to demand shocks
- However, adaptation alone does not guarantee safety
- LinUCB exhibits excessive panic and volatility
- Panic logic is reactive, not preventative

In this final section, we:
1. Introduce **meta-policies** that combine bandits safely
2. Treat safety as a **first-class architectural layer**
3. Re-run experiments and compare outcomes


## 1. Why Bandits Alone Are Not Enough

Empirical evidence from Part 4 shows:

- Overconfidence leads to unsafe pricing (LinUCB)
- Randomized exploration can cause spikes (Thompson)
- Panic logic reacts *after* damage occurs

Therefore, safety must:
- constrain exploration
- bound risk proactively
- operate *above* bandit logic

This motivates **meta-policies**.


## 2. Meta-Policy Design Principles

A valid combination must:

- Preserve exploration benefits
- Enforce conservative bounds
- Be interpretable and auditable
- Avoid naive averaging

We evaluate two designs:
1. Safety-Gated Thompson (All 3)
2. Enterprise Safe Mode (UCB + LinUCB)


## 3. Safety-Gated Thompson (All Three Combined)

Roles:
- Thompson → exploration & fast adaptation
- Bayesian UCB → optimistic upper bound
- LinUCB → conservative anchor

Final price is Thompson's proposal,
**clipped inside a safety envelope** defined by UCB and LinUCB.


In [ ]:
@dataclass
class PricingDecision:
    selected_price: float
    expected_revenue: float
    uncertainty_sigma: float
    source: str
    panic_mode: bool

In [ ]:
class SafetyGatedBandit:
    def __init__(self, thompson, ucb, linucb):
        self.thompson = thompson
        self.ucb = ucb
        self.linucb = linucb

    def choose_price(self, context, min_p=50, max_p=300):
        d_t = self.thompson.choose_price(context, min_p, max_p)
        d_u = self.ucb.choose_price(context, min_p, max_p)
        d_l = self.linucb.choose_price(context, min_p, max_p)

        lower = min(d_u.selected_price, d_l.selected_price)
        upper = max(d_u.selected_price, d_l.selected_price)
        final_price = np.clip(d_t.selected_price, lower, upper)

        self._last = (d_t, d_u, d_l)

        return PricingDecision(
            selected_price=final_price,
            expected_revenue=d_t.expected_revenue,
            uncertainty_sigma=d_t.uncertainty_sigma,
            source="SafetyGated(3)",
            panic_mode=d_t.panic_mode or d_u.panic_mode or d_l.panic_mode,
        )

    def update(self, context, decision, booked):
        d_t, d_u, d_l = self._last

        # Each bandit learns from ITS OWN action
        self.thompson.update(context, d_t.selected_price, booked)
        self.ucb.update(context, d_u.selected_price, booked)
        self.linucb.update(context, d_l.selected_price, booked)


In [ ]:
def run_simulation_2(agent, T=200, shock_time=100):
    history = []

    for t in range(T):
        decision = agent.choose_price(context, min_p=50, max_p=250)
        booked = market_response(decision.selected_price, t, shock_time)

        agent.update(context, decision, booked)

        history.append({
            "t": t,
            "price": decision.selected_price,
            "booked": booked,
            "panic": decision.panic_mode,
            "sigma": decision.uncertainty_sigma,
        })

    return pd.DataFrame(history)


## 4. Enterprise Safe Mode (UCB + LinUCB)

This mode is:
- deterministic
- conservative
- regulator-friendly

Used when:
- pricing risk is high
- exploration budget is limited
- explainability is mandatory


In [ ]:
class EnterpriseSafeBandit:
    def __init__(self, ucb, linucb, weight_ucb=0.7):
        self.ucb = ucb
        self.linucb = linucb
        self.w = weight_ucb

    def choose_price(self, context, min_p=50, max_p=300):
        d_u = self.ucb.choose_price(context, min_p, max_p)
        d_l = self.linucb.choose_price(context, min_p, max_p)

        self._last = (d_u, d_l)

        price = self.w * d_u.selected_price + (1 - self.w) * d_l.selected_price

        return PricingDecision(
            selected_price=price,
            expected_revenue=d_u.expected_revenue,
            uncertainty_sigma=d_u.uncertainty_sigma,
            source="Enterprise(UCB+LinUCB)",
            panic_mode=d_u.panic_mode or d_l.panic_mode,
        )

    def update(self, context, decision, booked):
        d_u, d_l = self._last
        self.ucb.update(context, d_u.selected_price, booked)
        self.linucb.update(context, d_l.selected_price, booked)


In [ ]:
meta_agents = {
    "SafetyGated(3)": SafetyGatedBandit(
        agents["Thompson"],
        agents["BayesianUCB"],
        agents["LinUCB"],
    ),
    "Enterprise(UCB+LinUCB)": EnterpriseSafeBandit(
        agents["BayesianUCB"],
        agents["LinUCB"],
    ),
}


In [ ]:
meta_results = {}

for name, agent in meta_agents.items():
    print(f"Running simulation for {name}...")
    meta_results[name] = run_simulation(agent)
